In [4]:

import numpy as np
import time

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ==========================================================
# 1. GENERATE CYBER ATTACK DETECTION DATASET
# ==========================================================

SEED = 42
np.random.seed(SEED)

X, y = make_classification(
    n_samples=2500,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    weights=[0.70, 0.30],
    class_sep=1.0,
    flip_y=0.01,
    random_state=SEED
)

feature_names = [f"Feature_{i+1}" for i in range(X.shape[1])]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=500,
    stratify=y,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=500,
    stratify=y_train_val,
    random_state=SEED
)

print("Cyber Attack Detection Dataset PSO")
print("-" * 55)
print("NAME: HEMALATHA S USN: 1BM25CS454")
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Total features:", X.shape[1])


# ==========================================================
# 2. PSO CONFIGURATION
# ==========================================================

N_FEATURES = X.shape[1]

POPULATION_SIZE = 10
MAX_ITERATIONS = 10

W = 0.7
C1 = 1.5
C2 = 1.5

# Particle dimensions:
# First 20 dimensions = feature selection
# Dimension 21 = number of trees
# Dimension 22 = maximum tree depth
# Dimension 23 = minimum samples to split

DIMENSIONS = N_FEATURES + 3

LOWER_BOUNDS = np.zeros(DIMENSIONS)
UPPER_BOUNDS = np.ones(DIMENSIONS)


# ==========================================================
# 3. DECODE PARTICLE POSITION
# ==========================================================

def decode_particle(position):

    selected = position[:N_FEATURES] >= 0.5

    if not np.any(selected):
        selected[np.argmax(position[:N_FEATURES])] = True

    n_estimators = int(round(
        50 + position[N_FEATURES] * 150
    ))

    max_depth = int(round(
        2 + position[N_FEATURES + 1] * 18
    ))

    min_samples_split = int(round(
        2 + position[N_FEATURES + 2] * 8
    ))

    n_estimators = int(np.clip(n_estimators, 50, 200))
    max_depth = int(np.clip(max_depth, 2, 20))
    min_samples_split = int(
        np.clip(min_samples_split, 2, 10)
    )

    return (
        selected,
        n_estimators,
        max_depth,
        min_samples_split
    )


# ==========================================================
# 4. FITNESS FUNCTION: VALIDATION F1-SCORE
# ==========================================================

def fitness_function(position):

    (
        selected_features,
        n_estimators,
        max_depth,
        min_samples_split
    ) = decode_particle(position)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train[:, selected_features],
        y_train
    )

    predictions = model.predict(
        X_val[:, selected_features]
    )

    score = f1_score(
        y_val,
        predictions,
        zero_division=0
    )

    return score


# ==========================================================
# 5. PARTICLE SWARM OPTIMIZATION
# ==========================================================

def particle_swarm_optimization():

    rng = np.random.default_rng(SEED)

    positions = rng.uniform(
        0, 1,
        size=(POPULATION_SIZE, DIMENSIONS)
    )

    velocities = rng.uniform(
        -0.1, 0.1,
        size=(POPULATION_SIZE, DIMENSIONS)
    )

    personal_best_positions = positions.copy()

    personal_best_scores = np.array([
        fitness_function(particle)
        for particle in positions
    ])

    best_index = np.argmax(personal_best_scores)

    global_best_position = (
        personal_best_positions[best_index].copy()
    )

    global_best_score = personal_best_scores[best_index]

    start_time = time.time()

    print("\nStarting Particle Swarm Optimization...\n")

    for iteration in range(MAX_ITERATIONS):

        for i in range(POPULATION_SIZE):

            r1 = rng.random(DIMENSIONS)
            r2 = rng.random(DIMENSIONS)

            velocities[i] = (
                W * velocities[i]
                + C1 * r1 * (
                    personal_best_positions[i] - positions[i]
                )
                + C2 * r2 * (
                    global_best_position - positions[i]
                )
            )

            positions[i] = np.clip(
                positions[i] + velocities[i],
                LOWER_BOUNDS,
                UPPER_BOUNDS
            )

            current_score = fitness_function(positions[i])

            if current_score > personal_best_scores[i]:

                personal_best_scores[i] = current_score

                personal_best_positions[i] = (
                    positions[i].copy()
                )

            if current_score > global_best_score:

                global_best_score = current_score

                global_best_position = (
                    positions[i].copy()
                )

        print(
            f"Iteration {iteration + 1:02d}/{MAX_ITERATIONS} "
            f"| Best Validation F1-score: "
            f"{global_best_score:.4f}"
        )

    execution_time = time.time() - start_time

    return (
        global_best_position,
        global_best_score,
        execution_time
    )


# ==========================================================
# 6. RUN PSO AND DISPLAY OPTIMIZATION RESULTS
# ==========================================================

best_position, best_fitness, pso_time = (
    particle_swarm_optimization()
)

(
    best_features,
    best_n_estimators,
    best_max_depth,
    best_min_samples_split
) = decode_particle(best_position)

print("\n" + "=" * 55)
print("BEST PSO OPTIMIZATION RESULTS")
print("=" * 55)

print(f"Best validation F1-score: {best_fitness:.4f}")

print("\nBest Hyperparameters:")
print("Number of trees:", best_n_estimators)
print("Maximum tree depth:", best_max_depth)
print("Minimum samples to split:", best_min_samples_split)

print("\nSelected Features:")

selected_feature_names = [
    feature_names[i]
    for i in range(N_FEATURES)
    if best_features[i]
]

for name in selected_feature_names:
    print(name)

print("\nNumber of selected features:", int(best_features.sum()))
print("Total available features:", N_FEATURES)

print(f"\nPSO execution time: {pso_time:.2f} seconds")


# ==========================================================
# 7. FINAL CYBER ATTACK DETECTION MODEL
# ==========================================================

X_final_train = X_train_val[:, best_features]
X_final_test = X_test[:, best_features]

final_model = RandomForestClassifier(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    min_samples_split=best_min_samples_split,
    random_state=SEED,
    n_jobs=-1
)

final_model.fit(X_final_train, y_train_val)

y_pred = final_model.predict(X_final_test)


# ==========================================================
# 8. EVALUATION METRICS
# ==========================================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred, zero_division=0
)
recall = recall_score(
    y_test, y_pred, zero_division=0
)
f1 = f1_score(
    y_test, y_pred, zero_division=0
)

print("\n" + "=" * 55)
print("CYBER ATTACK DETECTION PERFORMANCE")
print("=" * 55)

print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-score  : {f1 * 100:.2f}%")


# ==========================================================
# 9. CLASSIFICATION REPORT
# ==========================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=[
            "Normal Traffic",
            "Cyberattack"
        ],
        digits=4,
        zero_division=0
    )
)


# ==========================================================
# 10. CONFUSION MATRIX
# ==========================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1]
)

print("Confusion Matrix:")
print(cm)

print("\nConfusion Matrix Interpretation:")
print("True Negatives  (TN):", cm[0, 0])
print("False Positives (FP):", cm[0, 1])
print("False Negatives (FN):", cm[1, 0])
print("True Positives  (TP):", cm[1, 1])

print("\nExperiment completed successfully.")


Cyber Attack Detection Dataset PSO
-------------------------------------------------------
NAME: HEMALATHA S USN: 1BM25CS454
Training samples: 1500
Validation samples: 500
Testing samples: 500
Total features: 20

Starting Particle Swarm Optimization...

Iteration 01/10 | Best Validation F1-score: 0.8235
Iteration 02/10 | Best Validation F1-score: 0.8550
Iteration 03/10 | Best Validation F1-score: 0.8571
Iteration 04/10 | Best Validation F1-score: 0.8613
Iteration 05/10 | Best Validation F1-score: 0.8613
Iteration 06/10 | Best Validation F1-score: 0.8613
Iteration 07/10 | Best Validation F1-score: 0.8613
Iteration 08/10 | Best Validation F1-score: 0.8613
Iteration 09/10 | Best Validation F1-score: 0.8613
Iteration 10/10 | Best Validation F1-score: 0.8613

BEST PSO OPTIMIZATION RESULTS
Best validation F1-score: 0.8613

Best Hyperparameters:
Number of trees: 161
Maximum tree depth: 20
Minimum samples to split: 2

Selected Features:
Feature_1
Feature_2
Feature_3
Feature_4
Feature_6
Feature